## IMPUTATION OF MISSING GROUNDWATER LEVEL VALUES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer  # noqa
# from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from sklearn.model_selection import GridSearchCV

import shap
import os
import requests
import zipfile
import io

In [2]:
# since the .csv is large (>100 Mb) we download and read zip

# Google Drive file ID (from your original URL)
file_id = "11TvlNdqSQv6PPooHwt7f9A68Ibz1sQIU"

# Direct download link for Google Drive
zip_url = f"https://drive.google.com/uc?export=download&id={file_id}"

# Download and extract the ZIP file
response = requests.get(zip_url)
if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # Assume there's only one file in the ZIP
        file_name = z.namelist()[0]
        with z.open(file_name) as f:
            df = pd.read_csv(f)
else:
    print("Failed to download the ZIP file.")
df.head()

/tmp/ipython-input-2044147778.py:16: DtypeWarning: Columns (6,38,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


,date,well_no,precipitation_mm_eobs,temperature_C_eobs,actual_evapotranspiration_mm_gleam,potential_evapotranspiration_mm_gleam,site,local_ID,altitude_ m_asl,gw_level_m_asl,...,top_of_screen,bottom_of_screen,geometry,date_left,index_right,lat,lon,date_right,tws_mm_tavg_gldas,gws_mm_tavg_gldas
0,1950-01-01,LT_193,0.0,-8.88,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (24.619946942916 56.197237383388),NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1950-01-01,LT_202,0.0,-8.88,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (24.620043179747 56.197209951771),NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950-01-01,LT_216,0.0,-8.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (24.692295330589 56.205279710082),NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1950-01-01,LT_218,0.0,-8.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (24.697801139013 56.21381864295),NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1950-01-01,LT_220,0.0,-8.75,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,POINT (24.69305907439 56.20920104507),NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Define the list of well numbers
wells_of_interest = ['LT_35994', 'LT_35995', 'LT_35996', 'LV_12225', 'LV_22606', 'LV_22610',
                     'LT_220'
                     ]

# Ensure 'date' is datetime
df['date'] = pd.to_datetime(df['date'])

# Apply the initial filter
df = df[
    (df['well_no'].isin(wells_of_interest)) &
    (df['date'] >= pd.Timestamp('2003-01-01'))
]

# Feature Engineering

The core goal is to improve predictive performance by incorporating both immediate and lagged hydrometeorological signals, seasonal patterns, and anomaly detection. Below is an overview of the key feature groups:

1. Base Hydrometeorological and Groundwater Features
These include fundamental environmental variables such as precipitation, temperature, evapotranspiration, groundwater storage, and groundwater level. These variables are essential drivers and indicators of groundwater fluctuations.

2. Lagged Features
Groundwater responses often exhibit delayed reactions to surface conditions. Lagged versions of base features (e.g., 30, 60, 90, up to 360 days) capture these delayed effects, allowing the model to learn dependencies over varying time scales.

3. Rolling Statistics
Rolling window summaries (means or sums) smooth out short-term noise and highlight accumulated inputs like total precipitation or average temperature over defined periods (e.g., 60 or 90 days). These help model gradual changes and trends in the system.

4. Seasonal and Cyclical Features
Groundwater dynamics strongly depend on annual cycles driven by seasons. To capture this, sine and cosine transformations of day-of-year and month encode cyclical patterns without artificial discontinuities, allowing models to grasp seasonal timing and phase shifts.

5. Anomaly and Event Indicators
Features such as precipitation anomaly, drought flags, groundwater storage anomalies, and recharge event flags highlight deviations from normal conditions or rapid changes, essential for identifying stress periods and recharge episodes.

6. Cold Climate Flags
A permafrost flag derived from persistent below-freezing temperatures accounts for soil freezing effects that inhibit infiltration and groundwater recharge in cold regions.

7. Climatological Monthly Averages
Including long-term monthly averages provides baseline context, enabling the model to differentiate typical seasonal behavior from unusual conditions.

In [4]:

# --- Ensure datetime format in original DataFrame ---
# Converting the 'date' column to datetime ensures proper time-based operations (e.g., shifting, rolling, grouping)
df['date'] = pd.to_datetime(df['date'])

# --- Copy original df to preserve it ---
# Keeping backups of the original dataframe allows safe experimentation without altering source data
df_orig = df.copy()
df_work = df_orig.copy()

# --- Sort by well and date ---
# Ensures proper chronological order for each well, critical for generating lag and rolling time-series features
df_work = df_work.sort_values(by=['well_no', 'date'])

# --- Define base hydrometeorological and groundwater features ---
# These are key environmental inputs known to influence groundwater levels
base_features = [
    'precipitation_mm_eobs',              # Precipitation: major input to groundwater recharge
    'temperature_C_eobs',                 # Temperature: affects evapotranspiration and snowmelt
    'actual_evapotranspiration_mm_gleam', # Actual evapotranspiration: loss component affecting water balance
    'gws_mm_tavg_gldas',                  # Groundwater storage: represents historical memory of the system
    'gw_level_m_asl',                     # Groundwater level (target); also used for engineering more features
]

target_col = 'gw_level_m_asl'

# --- Define lags and rolling windows ---
# Lag: Captures delayed effects of meteorological conditions on groundwater
# Rolling: Captures smoothed trends or accumulated quantities
lag_days = [30, 60, 90, 180, 270, 360]
roll_days = [60, 90]

# --- Generate full list of expected feature names for reference/tracking ---
all_feature_names = []
for col in base_features:
    for lag in lag_days:
        all_feature_names.append(f'{col}_lag_{lag}d')
    for win in roll_days:
        if col in ['precipitation_mm_eobs', 'actual_evapotranspiration_mm_gleam']:
            all_feature_names.append(f'{col}_rollsum_{win}d')  # Sum important for cumulative inputs/losses
        else:
            all_feature_names.append(f'{col}_rollmean_{win}d') # Mean for smoothing continuous variables

# --- Feature engineering: lag and rolling stats ---
# These help capture time-dependent behaviors (memory) in the groundwater system
for col in base_features:
    for lag in lag_days:
        # Shift the feature backwards in time to simulate delayed influence
        df_work[f'{col}_lag_{lag}d'] = df_work.groupby('well_no')[col].shift(lag)
    for win in roll_days:
        if col == 'precipitation_mm_eobs':
            # Rolling sum over precipitation: approximates recharge potential
            df_work[f'{col}_rollsum_{win}d'] = df_work.groupby('well_no')[col].transform(
                lambda x: x.rolling(window=win, min_periods=1).sum()
            )
        else:
            # Rolling mean: helps smooth short-term variability (e.g., GWS, temperature)
            df_work[f'{col}_rollmean_{win}d'] = df_work.groupby('well_no')[col].transform(
                lambda x: x.rolling(window=win, min_periods=1).mean()
            )

# --- Time-based cyclical features ---
# Captures annual seasonality and periodic trends

# Day of year: numeric representation of position in the year
df_work['dayofyear'] = df_work['date'].dt.dayofyear

# Sine and cosine transformations allow ML models to learn seasonal cycles without sharp discontinuities
df_work['sin_doy'] = np.sin(2 * np.pi * df_work['dayofyear'] / 365.25)
df_work['cos_doy'] = np.cos(2 * np.pi * df_work['dayofyear'] / 365.25)

# Similarly, extract and transform calendar month
df_work['month'] = df_work['date'].dt.month
df_work['sin_month'] = np.sin(2 * np.pi * df_work['month'] / 12)
df_work['cos_month'] = np.cos(2 * np.pi * df_work['month'] / 12)

# Redundant annual signal representation (alternative view)
df_work['sin_annual'] = np.sin(2 * np.pi * df_work['date'].dt.dayofyear / 365.25)
df_work['cos_annual'] = np.cos(2 * np.pi * df_work['date'].dt.dayofyear / 365.25)

# --- Seasonal phase (circular angle) representation ---
# Captures timing of the seasonal cycle, useful for detecting phase shifts
df_work['seasonal_phase'] = np.arctan2(df_work['sin_doy'], df_work['cos_doy'])

# --- Precipitation anomaly (deviation from annual average) ---
# Important for identifying drought or wet periods relative to typical long-term conditions
df_work['precip_anomaly'] = df_work['precipitation_mm_eobs_rollsum_90d'] - \
                            df_work.groupby('well_no')['precipitation_mm_eobs'].transform(
                                lambda x: x.rolling(365, min_periods=180).mean()
                            )

# Binary drought indicator
# Helps the model identify stress periods that may suppress groundwater recharge
df_work['drought_flag'] = df_work['precip_anomaly'] < -1.5

# --- Groundwater storage (GWS) monthly anomaly ---
# Detects deviation from average seasonal behavior
df_work['gws_monthly_mean'] = df_work.groupby(df_work['date'].dt.month)['gws_mm_tavg_gldas'].transform('mean')
df_work['gws_monthly_anomaly'] = df_work['gws_mm_tavg_gldas'] - df_work['gws_monthly_mean']

# --- Recharge detection ---
# Identifies rapid groundwater increases, possibly tied to rainfall/recharge events
df_work['gws_delta_30d'] = df_work['gws_mm_tavg_gldas'] - df_work['gws_mm_tavg_gldas_lag_30d']
df_work['gws_recharge_event'] = df_work['gws_delta_30d'] > df_work['gws_delta_30d'].quantile(0.9)

# --- Permafrost flag based on freezing temperature persistence ---
# Useful in cold climates where frozen soil inhibits infiltration
df_work['below_zero_rolling_30d'] = (
    df_work['temperature_C_eobs'].rolling(window=30, min_periods=30)
    .apply(lambda x: np.mean(x < 0), raw=True)
)
# Binary flag: 80% or more of the past 30 days were freezing
df_work['permafrost_flag'] = df_work['below_zero_rolling_30d'] > 0.8

# --- Climatological monthly averages (used for baseline comparisons) ---
# Adds contextual reference to understand if current values are anomalous
monthly_avg = df_work.groupby(df_work['date'].dt.month).agg({
    'temperature_C_eobs': 'mean',
    'precipitation_mm_eobs': 'mean',
    'gws_mm_tavg_gldas': 'mean'
}).rename(columns={
    'temperature_C_eobs': 'temp_monthly_avg',
    'precipitation_mm_eobs': 'precip_monthly_avg',
    'gws_mm_tavg_gldas': 'gws_monthly_avg'
})

# Map climatological averages back to main dataframe based on calendar month
df_work['temp_monthly_avg'] = df_work['month'].map(monthly_avg['temp_monthly_avg'])
df_work['precip_monthly_avg'] = df_work['month'].map(monthly_avg['precip_monthly_avg'])
df_work['gws_monthly_avg'] = df_work['month'].map(monthly_avg['gws_monthly_avg'])

## Signal cleaning/procesing

In [5]:
# Preserve the original groundwater level data before cleaning
df_work['gw_level_m_asl_original'] = df_work['gw_level_m_asl']

# Sort the dataframe by well number and date to ensure time-ordering for accurate differencing
df_work = df_work.sort_values(by=['well_no', 'date'])

# Calculate the absolute difference in groundwater level between each row and the previous row (forward diff)
diff_forward = df_work.groupby('well_no')['gw_level_m_asl'].diff().abs()

# Calculate the absolute difference between each row and the next row (backward diff)
diff_backward = df_work.groupby('well_no')['gw_level_m_asl'].diff(-1).abs()

# Define a threshold to identify unusually steep changes (outliers).
# Here, the 99.95th percentile of all absolute changes is used as a dynamic threshold.
threshold = df_work['gw_level_m_asl'].diff().abs().quantile(0.9995)

# Create a boolean mask to flag all values that show a sharp increase or decrease
# Either forward or backward difference exceeding the threshold is considered a steep jump
steep_mask = (diff_forward > threshold) | (diff_backward > threshold)

# Replace the identified steep peak values in the main signal column with NaN
# This removes probable noise, spikes, or sensor errors while preserving trend continuity
df_work.loc[steep_mask, 'gw_level_m_asl'] = np.nan

# Drop all data from June 1, 2022 onwards for well 'LT_35996' as it is known to be corrupted
df_work.loc[(df_work['well_no'] == 'LT_35996') & (df_work['date'] >= pd.Timestamp('2022-06-01')), 'gw_level_m_asl'] = np.nan

# Drop all data from January 1, 2014 onwards for well 'LT_220' as it is known to be corrupted
# df_work.loc[(df_work['well_no'] == 'LT_220') & (df_work['date'] <= pd.Timestamp('2014-01-01')), 'gw_level_m_asl'] = np.nan

# add the average 12-31 gw_level value for each well_no for succesful inputation

# Step 1: Compute average gw_level for each well_no on all past Dec 31 dates (any year)
dec_31_df = df_work[(df_work['date'].dt.month == 12) & (df_work['date'].dt.day == 31)]
avg_gw_levels = dec_31_df.groupby('well_no')['gw_level_m_asl'].mean()

# Step 2: Update only existing rows for 2023-12-31 where gw_level is NaN
mask = (df_work['date'] == pd.Timestamp('2023-12-31')) & (df_work['gw_level_m_asl'].isna())

# Loop through those rows and update
for idx in df_work[mask].index:
    well = df_work.loc[idx, 'well_no']
    if well in avg_gw_levels:
        df_work.at[idx, 'gw_level_m_asl'] = avg_gw_levels[well]

## Define Feature Sets

Although a rich set of features is engineered to capture all relevant dynamics, many of these will be reduced or dropped later due to multicollinearity. This is done using Variance Inflation Factor (VIF) filtering, aiming to retain only those features with VIF < 5, ensuring the model remains stable, generalizable, and interpretable.

In [6]:
# --- Manually selected features for modeling ---
# Define feature sets
feature_sets = {
    'with_seasonal': [
        'precipitation_mm_eobs_rollsum_90d',
        # 'temperature_C_eobs',
        'temperature_C_eobs_lag_30d',
        # 'temperature_C_eobs_lag_60d',
        'temperature_C_eobs_lag_90d',
        'temperature_C_eobs_lag_180d',
        'temperature_C_eobs_lag_270d',
        'temperature_C_eobs_lag_360d',
        # 'gws_mm_tavg_gldas_lag_30d',
        # 'gws_mm_tavg_gldas_lag_60d',
        'gws_mm_tavg_gldas_lag_90d',
        'gws_mm_tavg_gldas_lag_180d',
        'gws_mm_tavg_gldas_lag_270d',
        'gws_mm_tavg_gldas_lag_360d',
        'gws_mm_tavg_gldas',
        # 'gws_recharge_event',
        # 'permafrost_flag',
        # 'sin_month',
        # 'cos_month',
        # 'sin_annual',
        # 'cos_annual',
        # 'sin_doy',
        # 'cos_doy',
        'seasonal_phase',
        # 'temp_monthly_avg',
        'precip_monthly_avg',
        # 'gws_monthly_avg',
    ],
    'no_seasonal': [
        'precipitation_mm_eobs_rollsum_90d',
        # 'temperature_C_eobs',
        'temperature_C_eobs_lag_30d',
        # 'temperature_C_eobs_lag_60d',
        'temperature_C_eobs_lag_90d',
        'temperature_C_eobs_lag_180d',
        'temperature_C_eobs_lag_270d',
        'temperature_C_eobs_lag_360d',
        # 'gws_mm_tavg_gldas_lag_30d',
        # 'gws_mm_tavg_gldas_lag_60d',
        'gws_mm_tavg_gldas_lag_90d',
        'gws_mm_tavg_gldas_lag_180d',
        'gws_mm_tavg_gldas_lag_270d',
        'gws_mm_tavg_gldas_lag_360d',
        'gws_mm_tavg_gldas'
    ],
}

## Modeling
Hyperparameter selection basis:
* n_estimators – Number of Trees. Controls the ensemble size. More trees usually improve performance up to a point but increase compute cost. 100 is a solid baseline; 200–300 helps reduce variance for noisy groundwater data.
* max_depth. None (default) lets trees grow fully — may overfit. Capping depth (e.g., 10, 20, 30) helps avoid overfitting on seasonal or short-term fluctuations in the groundwater signal.
* min_samples_split. Forces the model to make splits only when there’s enough data. Higher values (e.g., 5 or 10) reduce overfitting by preventing too many small branches.
* min_samples_leaf. Prevents overly specific leaves based on a few samples.Important for time series or geophysical data where smoothing is often beneficial.
* max_features. Controls feature randomness: 'auto' or 'sqrt' can improve diversity in trees (better generalization). 0.5 tries only half the features at each split — useful if many input features are correlated (common in GW applications).



In [7]:
# --- Config ---
target_col = 'gw_level_m_asl'
min_samples_required = 5
min_features_ratio = 0.5
eval_fraction = 0.2
cv_fraction = 0.2
cv_repeats = 5
min_mask_length = 7
max_mask_length = 60

param_grid = {
    'n_estimators': [100, 200],              # Number of trees in the forest
    'max_depth': [None, 20],              # Controls overfitting
    'min_samples_split': [2],              # Minimum samples to split an internal node
    'min_samples_leaf': [1, 2],                # Minimum samples at a leaf node
}

models = {
    'ExtraTreesTuned': GridSearchCV(
        ExtraTreesRegressor(random_state=42),
        param_grid=param_grid,
        cv=3,
        scoring='r2',
        n_jobs=-1
    )
}

# --- Utility ---
def generate_random_intervals(data, total_target_count, min_len=7, max_len=30, max_attempts=1000):
    data = data.reset_index()
    selected_idx = set()
    attempts = 0
    while len(selected_idx) < total_target_count and attempts < max_attempts:
        interval_len = np.random.randint(min_len, max_len + 1)
        if len(data) <= interval_len:
            break
        start_idx = np.random.randint(0, len(data) - interval_len)
        candidate = data.iloc[start_idx:start_idx + interval_len]['index'].tolist()
        if not selected_idx.intersection(candidate):
            selected_idx.update(candidate)
        attempts += 1
    return list(selected_idx)


In [8]:
# --- Storage ---
imputed_all_models = {}
cv_scores_all = {}
skipped_wells = []
feature_importances_all = {}
vif_all = {}
best_model_per_well = {}
best_model_params = {}
cv_masked_intervals = {}

# --- Modeling + Imputation Loop (with GridSearchCV) ---

np.random.seed(42)
for well_no, group in df_work.groupby('well_no'):
    group = group.copy()
    group['date'] = pd.to_datetime(group['date'])

    # Skip wells with no valid targets
    if not group[target_col].notnull().any():
        skipped_wells.append(well_no)
        continue

    # Trim to first/last non-null target entries
    group = group[(group['date'] >= group[group[target_col].notnull()]['date'].min()) &
                  (group['date'] <= group[group[target_col].notnull()]['date'].max())]

    data_non_null = group[group[target_col].notnull()].sort_values('date')
    if len(data_non_null) < min_samples_required:
        skipped_wells.append(well_no)
        continue

    # Mask a portion for evaluation
    eval_idx = generate_random_intervals(data_non_null,
                                         total_target_count=int(eval_fraction * len(data_non_null)),
                                         min_len=min_mask_length, max_len=max_mask_length)
    # group['_was_masked'] = False
    group.loc[:, '_was_masked'] = False
    group.loc[eval_idx, '_was_masked'] = True

    # Track best
    best_r2 = -np.inf
    best_key = None
    best_params = None

    for feature_set_name, selected_features in feature_sets.items():
        feature_candidates = [f for f in selected_features if f in group.columns]
        if len(feature_candidates) < 2:
            continue

        observed_data = group[~group['_was_masked'] & group[target_col].notnull()].copy()
        if len(observed_data) < min_samples_required:
            continue

        min_required = int(len(feature_candidates) * min_features_ratio)

        for model_name, model_proto in models.items():
            key = (model_name, feature_set_name)
            cv_scores_all.setdefault(key, [])
            imputed_all_models.setdefault(key, [])
            feature_importances_all.setdefault(key, [])

            r2_list, mae_list = [], []

            for _ in range(cv_repeats):
                mask_idx = generate_random_intervals(
                    observed_data,
                    total_target_count=int(cv_fraction * len(observed_data)),
                    min_len=min_mask_length,
                    max_len=max_mask_length
                )
                if _ == 0:
                  mask_dates = observed_data.loc[mask_idx, 'date'].sort_values()
                  if not mask_dates.empty:
                      gaps = mask_dates.diff().dt.days.fillna(1)
                      segment_ids = (gaps > 1).cumsum()
                      intervals = []
                      for _, segment in mask_dates.groupby(segment_ids):
                          intervals.append((segment.min(), segment.max()))
                      cv_masked_intervals[(well_no, model_name, feature_set_name)] = intervals

                train_cv = observed_data.drop(index=mask_idx)
                test_cv = observed_data.loc[mask_idx]

                train_cv = train_cv.dropna(subset=feature_candidates, thresh=min_required)
                test_cv = test_cv.dropna(subset=feature_candidates, thresh=min_required)
                if train_cv.empty or test_cv.empty:
                    continue

                # Prepare data
                fill_values = train_cv[feature_candidates].median()
                X_train = train_cv[feature_candidates].fillna(fill_values)
                X_test = test_cv[feature_candidates].fillna(fill_values)

                scaler = StandardScaler().fit(X_train)
                X_train_scaled = scaler.transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                # Fit and predict
                model = model_proto.fit(X_train_scaled, train_cv[target_col])
                preds = model.predict(X_test_scaled)

                r2_list.append(r2_score(test_cv[target_col], preds))
                mae_list.append(mean_absolute_error(test_cv[target_col], preds))

            if r2_list:
                r2_mean = np.mean(r2_list)
                mae_mean = np.mean(mae_list)
                cv_scores_all[key].append((well_no, r2_mean, np.std(r2_list), mae_mean, np.std(mae_list)))

                if r2_mean > best_r2:
                    best_r2 = r2_mean
                    best_key = key
                    best_params = model.best_params_

    if best_key is None:
        skipped_wells.append(well_no)
        continue

    # Train final model on all available data
    model_name, feature_set_name = best_key
    feature_candidates = [f for f in feature_sets[feature_set_name] if f in group.columns]
    min_required = int(len(feature_candidates) * min_features_ratio)

    observed_data = group[~group['_was_masked'] & group[target_col].notnull()].copy()
    fill_values_vif = observed_data[feature_candidates].median()
    scaled_vif_data = observed_data[feature_candidates].fillna(fill_values_vif)
    scaled_vif = StandardScaler().fit_transform(scaled_vif_data)
    vif_scores = [variance_inflation_factor(scaled_vif, i) for i in range(len(feature_candidates))]
    vif_all.setdefault(feature_set_name, []).append(dict(zip(feature_candidates, vif_scores)))

    full_train = group[group[target_col].notnull()]
    full_predict = group[group[target_col].isnull()]
    full_train = full_train.dropna(subset=feature_candidates, thresh=min_required)
    full_predict = full_predict.dropna(subset=feature_candidates, thresh=min_required)
    if full_train.empty or full_predict.empty:
        continue

    fill_values_full = full_train[feature_candidates].median()
    X_train = full_train[feature_candidates].fillna(fill_values_full)
    X_predict = full_predict[feature_candidates].fillna(fill_values_full)

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_predict_scaled = scaler.transform(X_predict)

    final_model = ExtraTreesRegressor(random_state=42, **best_params)
    final_model.fit(X_train_scaled, full_train[target_col])
    preds = final_model.predict(X_predict_scaled)

    group_copy = group.copy()
    group_copy.loc[full_predict.index, target_col] = preds
    group_copy['feature_set'] = feature_set_name
    imputed_all_models[best_key].append(group_copy)

    if hasattr(final_model, 'feature_importances_'):
        importances_dict = dict(zip(feature_candidates, final_model.feature_importances_))
        feature_importances_all[best_key].append(importances_dict)

    best_model_per_well[well_no] = best_key
    best_model_params[well_no] = best_params

    rows = []
for well_no, best_key in best_model_per_well.items():
    model_name, feature_set_name = best_key
    # find the CV metrics for this well under the winning key
    metrics = next(
        (t for t in cv_scores_all.get(best_key, []) if t[0] == well_no),
        None
    )
    if metrics is None:
        r2_mean = r2_std = mae_mean = mae_std = float('nan')
    else:
        _, r2_mean, r2_std, mae_mean, mae_std = metrics

    rows.append({
        "well_no": well_no,
        "CV_R2 mean": r2_mean,
        "CV_R2 std": r2_std,
        "CV_MAE mean": mae_mean,
        "CV_MAE std": mae_std,
        "feature_set_name": feature_set_name,
        "best_params": best_model_params.get(well_no),
        # Optional: include model_name if you want to see it
        # "model_name": model_name,
    })

import pandas as pd
summary = pd.DataFrame(rows).sort_values("CV_R2 mean", ascending=False)

# Add an AVERAGE row like in your example
avg = {
    "well_no": "AVERAGE",
    "CV_R2 mean": summary["CV_R2 mean"].mean(),
    "CV_R2 std": summary["CV_R2 std"].mean(),
    "CV_MAE mean": summary["CV_MAE mean"].mean(),
    "CV_MAE std": summary["CV_MAE std"].mean(),
    "feature_set_name": "",
    "best_params": "",
}
summary = pd.concat([summary, pd.DataFrame([avg])], ignore_index=True)

print(summary.to_string(index=False))

 well_no  CV_R2 mean  CV_R2 std  CV_MAE mean  CV_MAE std feature_set_name                                                                             best_params
LT_35994    0.790482   0.076180     0.157827    0.012348    with_seasonal {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
LV_12225    0.673033   0.075276     0.117519    0.012208      no_seasonal   {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
LT_35995    0.643024   0.083962     0.122320    0.014305      no_seasonal   {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
  LT_220    0.576860   0.129742     0.170395    0.023612      no_seasonal {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
LV_22606    0.575158   0.126216     0.158444    0.014447    with_seasonal {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
LV_22610    0.488683   0.089

In [ ]:
# The same as above only with shap

# --- Imports (add these if not already present) ---
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
import shap

# --- Storage ---
imputed_all_models = {}
cv_scores_all = {}
skipped_wells = []
feature_importances_all = {}
vif_all = {}
best_model_per_well = {}
best_model_params = {}
cv_masked_intervals = {}

# NEW: SHAP storage
#   key is (model_name, feature_set_name)
#   each value is a list of dicts, 1 dict per well that used this key as the winner
shap_values_all = {}          # stores raw SHAP arrays per well (possibly sampled)
shap_importances_all = {}     # stores per-well mean|SHAP| dicts (feature->importance)
shap_metadata_all = {}        # stores ancillary info to help with plotting/traceability

# For memory/runtime hygiene: sample at most this many rows per well for SHAP
MAX_SHAP_SAMPLES_PER_WELL = 1000
RANDOM_STATE = 42

# --- Modeling + Imputation Loop (with GridSearchCV) ---
np.random.seed(RANDOM_STATE)
for well_no, group in df_work.groupby('well_no'):
    group = group.copy()
    group['date'] = pd.to_datetime(group['date'])

    # Skip wells with no valid targets
    if not group[target_col].notnull().any():
        skipped_wells.append(well_no)
        continue

    # Trim to first/last non-null target entries
    group = group[(group['date'] >= group[group[target_col].notnull()]['date'].min()) &
                  (group['date'] <= group[group[target_col].notnull()]['date'].max())]

    data_non_null = group[group[target_col].notnull()].sort_values('date')
    if len(data_non_null) < min_samples_required:
        skipped_wells.append(well_no)
        continue

    # Mask a portion for evaluation
    eval_idx = generate_random_intervals(
        data_non_null,
        total_target_count=int(eval_fraction * len(data_non_null)),
        min_len=min_mask_length,
        max_len=max_mask_length
    )
    group.loc[:, '_was_masked'] = False
    group.loc[eval_idx, '_was_masked'] = True

    # Track best
    best_r2 = -np.inf
    best_key = None
    best_params = None

    for feature_set_name, selected_features in feature_sets.items():
        feature_candidates = [f for f in selected_features if f in group.columns]
        if len(feature_candidates) < 2:
            continue

        observed_data = group[~group['_was_masked'] & group[target_col].notnull()].copy()
        if len(observed_data) < min_samples_required:
            continue

        min_required = int(len(feature_candidates) * min_features_ratio)

        for model_name, model_proto in models.items():
            key = (model_name, feature_set_name)
            cv_scores_all.setdefault(key, [])
            imputed_all_models.setdefault(key, [])
            feature_importances_all.setdefault(key, [])

            r2_list, mae_list = [], []

            for rep in range(cv_repeats):
                mask_idx = generate_random_intervals(
                    observed_data,
                    total_target_count=int(cv_fraction * len(observed_data)),
                    min_len=min_mask_length,
                    max_len=max_mask_length
                )
                if rep == 0:
                    mask_dates = observed_data.loc[mask_idx, 'date'].sort_values()
                    if not mask_dates.empty:
                        gaps = mask_dates.diff().dt.days.fillna(1)
                        segment_ids = (gaps > 1).cumsum()
                        intervals = []
                        for _, segment in mask_dates.groupby(segment_ids):
                            intervals.append((segment.min(), segment.max()))
                        cv_masked_intervals[(well_no, model_name, feature_set_name)] = intervals

                train_cv = observed_data.drop(index=mask_idx)
                test_cv = observed_data.loc[mask_idx]

                train_cv = train_cv.dropna(subset=feature_candidates, thresh=min_required)
                test_cv = test_cv.dropna(subset=feature_candidates, thresh=min_required)
                if train_cv.empty or test_cv.empty:
                    continue

                # Prepare data
                fill_values = train_cv[feature_candidates].median()
                X_train = train_cv[feature_candidates].fillna(fill_values)
                X_test = test_cv[feature_candidates].fillna(fill_values)

                scaler = StandardScaler().fit(X_train)
                X_train_scaled = scaler.transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                # Fit and predict (model_proto is assumed to be a GridSearchCV instance)
                model = model_proto.fit(X_train_scaled, train_cv[target_col])
                preds = model.predict(X_test_scaled)

                r2_list.append(r2_score(test_cv[target_col], preds))
                mae_list.append(mean_absolute_error(test_cv[target_col], preds))

            if r2_list:
                r2_mean = np.mean(r2_list)
                mae_mean = np.mean(mae_list)
                cv_scores_all[key].append((well_no, r2_mean, np.std(r2_list), mae_mean, np.std(mae_list)))

                if r2_mean > best_r2:
                    best_r2 = r2_mean
                    best_key = key
                    # Save best params from the GridSearchCV
                    best_params = model.best_params_

    if best_key is None:
        skipped_wells.append(well_no)
        continue

    # Train final model on all available data for the winning key
    model_name, feature_set_name = best_key
    feature_candidates = [f for f in feature_sets[feature_set_name] if f in group.columns]
    min_required = int(len(feature_candidates) * min_features_ratio)

    observed_data = group[~group['_was_masked'] & group[target_col].notnull()].copy()
    fill_values_vif = observed_data[feature_candidates].median()
    scaled_vif_data = observed_data[feature_candidates].fillna(fill_values_vif)
    scaled_vif = StandardScaler().fit_transform(scaled_vif_data)
    vif_scores = [variance_inflation_factor(scaled_vif, i) for i in range(len(feature_candidates))]
    vif_all.setdefault(feature_set_name, []).append(dict(zip(feature_candidates, vif_scores)))

    full_train = group[group[target_col].notnull()]
    full_predict = group[group[target_col].isnull()]
    full_train = full_train.dropna(subset=feature_candidates, thresh=min_required)
    full_predict = full_predict.dropna(subset=feature_candidates, thresh=min_required)
    if full_train.empty:
        skipped_wells.append(well_no)
        continue

    # Prepare final train/predict matrices
    fill_values_full = full_train[feature_candidates].median()
    X_train = full_train[feature_candidates].fillna(fill_values_full)
    y_train = full_train[target_col].values
    X_predict = full_predict[feature_candidates].fillna(fill_values_full) if not full_predict.empty else None

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_predict_scaled = scaler.transform(X_predict) if X_predict is not None else None

    # Final model uses the best hyperparams found
    final_model = ExtraTreesRegressor(random_state=RANDOM_STATE, **best_params)
    final_model.fit(X_train_scaled, y_train)

    # Impute predictions for null targets
    if X_predict is not None and len(X_predict) > 0:
        preds = final_model.predict(X_predict_scaled)
        group_copy = group.copy()
        group_copy.loc[full_predict.index, target_col] = preds
        group_copy['feature_set'] = feature_set_name
        imputed_all_models[best_key].append(group_copy)

    # Save native model feature importances (tree-based)
    if hasattr(final_model, 'feature_importances_'):
        importances_dict = dict(zip(feature_candidates, final_model.feature_importances_))
        feature_importances_all[best_key].append(importances_dict)

    # Record winners
    best_model_per_well[well_no] = best_key
    best_model_params[well_no] = best_params

    # --------- NEW: SHAP collection for the final (winning) model ----------
    # Sample rows for SHAP if needed (to keep memory reasonable)
    rng = np.random.default_rng(RANDOM_STATE)
    if len(X_train) > MAX_SHAP_SAMPLES_PER_WELL:
        sample_idx = rng.choice(len(X_train), size=MAX_SHAP_SAMPLES_PER_WELL, replace=False)
        X_train_sampled = X_train.iloc[sample_idx].reset_index(drop=True)
        X_train_scaled_sampled = X_train_scaled[sample_idx, :]
        y_train_sampled = y_train[sample_idx]
    else:
        X_train_sampled = X_train.reset_index(drop=True)
        X_train_scaled_sampled = X_train_scaled
        y_train_sampled = y_train

    # TreeExplainer works well for ExtraTrees
    explainer = shap.TreeExplainer(final_model)
    shap_vals = explainer.shap_values(X_train_scaled_sampled)
    base_val = explainer.expected_value

    # Persist SHAP arrays + lightweight summaries
    shap_values_all.setdefault(best_key, []).append({
        "well_no": well_no,
        "feature_names": feature_candidates,
        "shap_values": shap_vals,                              # np.ndarray [n_samples, n_features]
        "base_value": base_val,                                # float or np.ndarray
        "X_features": X_train_sampled[feature_candidates],     # unscaled features for nicer plots
        "y": y_train_sampled,                                  # target (sampled)
    })

    mean_abs = np.abs(shap_vals).mean(axis=0)
    shap_importances_all.setdefault(best_key, []).append(dict(zip(feature_candidates, mean_abs)))

    shap_metadata_all.setdefault(best_key, []).append({
        "well_no": well_no,
        "n_samples": len(X_train_sampled),
        "feature_set_name": feature_set_name,
        "model_name": model_name,
        "fill_values": fill_values_full.to_dict(),
        "scaler_type": "StandardScaler",
    })
    # ----------------------------------------------------------------------

# --- Build summary table (unchanged) ---
rows = []
for well_no, best_key in best_model_per_well.items():
    model_name, feature_set_name = best_key
    metrics = next((t for t in cv_scores_all.get(best_key, []) if t[0] == well_no), None)
    if metrics is None:
        r2_mean = r2_std = mae_mean = mae_std = float('nan')
    else:
        _, r2_mean, r2_std, mae_mean, mae_std = metrics

    rows.append({
        "well_no": well_no,
        "CV_R2 mean": r2_mean,
        "CV_R2 std": r2_std,
        "CV_MAE mean": mae_mean,
        "CV_MAE std": mae_std,
        "feature_set_name": feature_set_name,
        "best_params": best_model_params.get(well_no),
        # "model_name": model_name,  # optional
    })

summary = pd.DataFrame(rows).sort_values("CV_R2 mean", ascending=False)

# Add an AVERAGE row
avg = {
    "well_no": "AVERAGE",
    "CV_R2 mean": summary["CV_R2 mean"].mean(),
    "CV_R2 std": summary["CV_R2 std"].mean(),
    "CV_MAE mean": summary["CV_MAE mean"].mean(),
    "CV_MAE std": summary["CV_MAE std"].mean(),
    "feature_set_name": "",
    "best_params": "",
}
summary = pd.concat([summary, pd.DataFrame([avg])], ignore_index=True)

print(summary.to_string(index=False))


In [ ]:
# --- Stacked SHAP beeswarms (single JPEG output, 600 dpi) ---
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Default font sizes (can all be overridden via the fonts dict or font_scale)
SUBPLOT_LABEL_FS = 16  # label like "(a)", "(b)", ...

def _index_to_alpha_label(idx: int) -> str:
    """
    0 -> (a), 1 -> (b), ..., 25 -> (z), 26 -> (aa), 27 -> (ab), ...
    """
    letters = []
    n = idx
    while True:
        letters.append(chr(97 + (n % 26)))
        n = n // 26 - 1
        if n < 0:
            break
    return f"({''.join(reversed(letters))})"


def _concat_shap_for_feature_set(shap_values_all, feature_set_name, limit_wells=None):
    """
    Concatenate SHAP arrays and feature matrices across wells for a given feature set.
    Accepts either raw numpy arrays or shap.Explanation objects in rec["shap_values"].
    """
    shap_arrays, feature_frames = [], []
    used = 0

    for (model_name, fs), records in shap_values_all.items():
        if fs != feature_set_name:
            continue
        for rec in records:
            sv = rec["shap_values"]
            if hasattr(sv, "values"):  # Explanation object
                sv = sv.values
            shap_arrays.append(sv)
            feature_frames.append(rec["X_features"][rec["feature_names"]])
            used += 1
            if limit_wells is not None and used >= limit_wells:
                break
        if limit_wells is not None and used >= limit_wells:
            break

    if not shap_arrays:
        return None, None, None

    shap_mat = np.vstack(shap_arrays)
    features_df = pd.concat(feature_frames, axis=0, ignore_index=True)
    feature_names = list(features_df.columns)
    return shap_mat, features_df, feature_names


def _apply_font_sizes_to_axes(fig, fonts):
    """
    After SHAP draws the plot, sweep through axes and apply sizes so it works across
    SHAP/matplotlib versions (including colorbar axis).
    """
    if not fonts:
        return

    tick_fs = fonts.get("tick")
    axis_label_fs = fonts.get("axis_label")
    cbar_tick_fs = fonts.get("cbar_tick", tick_fs)      # fallback to tick size
    cbar_label_fs = fonts.get("cbar_label", axis_label_fs)

    for ax in fig.axes:
        # Tick labels
        if tick_fs is not None:
            ax.tick_params(axis="both", which="both", labelsize=tick_fs)
        # Axis labels
        if axis_label_fs is not None:
            ax.xaxis.label.set_size(axis_label_fs)
            ax.yaxis.label.set_size(axis_label_fs)

        # Heuristic: colorbar axis often has no images but does have ticks/label
        if hasattr(ax, "get_images") and len(ax.get_images()) == 0:
            if cbar_tick_fs is not None:
                ax.tick_params(axis="both", which="both", labelsize=cbar_tick_fs)
            if cbar_label_fs is not None:
                ax.yaxis.label.set_size(cbar_label_fs)


def _rc_updates_from_fonts(fonts, font_scale=1.0):
    """
    Build a dict of rcParams updates from a fonts dict and scale factor.
    Keys you can use in fonts: base, title, axis_label, tick, legend, suptitle
    """
    if fonts is None:
        fonts = {}

    def scaled(val):
        return None if val is None else float(val) * float(font_scale)

    rc = {}
    base = scaled(fonts.get("base"))
    title = scaled(fonts.get("title"))
    axis_label = scaled(fonts.get("axis_label"))
    tick = scaled(fonts.get("tick"))
    legend = scaled(fonts.get("legend"))
    suptitle = scaled(fonts.get("suptitle"))

    if base is not None:
        rc["font.size"] = base
    if title is not None:
        rc["axes.titlesize"] = title
    if axis_label is not None:
        rc["axes.labelsize"] = axis_label
    if tick is not None:
        rc["xtick.labelsize"] = tick
        rc["ytick.labelsize"] = tick
    if legend is not None:
        rc["legend.fontsize"] = legend
    if suptitle is not None:
        rc["figure.titlesize"] = suptitle

    return rc


def _render_beeswarm_to_png_bytes(
    shap_mat,
    features_df,
    feature_names,
    title,
    max_display=20,
    rng=None,
    panel_dpi=600,
    fonts=None,
    font_scale=1.0,
    xlim=None,  # NEW: force x-axis range (e.g., (-1, 1))
):
    """
    Render a single shap.summary_plot to in-memory PNG bytes (high DPI), then return the PNG bytes.
    Using rng avoids the FutureWarning and makes jitter reproducible.

    Font control:
    - Pass a dict fonts with any of: base, title, axis_label, tick, legend, suptitle, cbar_tick, cbar_label
    - Optionally multiply all these via font_scale (float).

    xlim:
    - If provided, forces the main data axis limits before rasterization (keeps panels comparable).
    """
    rc_updates = _rc_updates_from_fonts(fonts, font_scale=font_scale)

    # Draw the summary plot without showing it, inside a temporary rc context
    with plt.rc_context(rc=rc_updates):
        try:
            shap.summary_plot(
                shap_mat,
                features=features_df,
                feature_names=feature_names,
                max_display=max_display,
                show=False,
                rng=rng,  # newer SHAP supports this
            )
        except TypeError:
            # Backward-compat: SHAP without rng kwarg
            shap.summary_plot(
                shap_mat,
                features=features_df,
                feature_names=feature_names,
                max_display=max_display,
                show=False,
            )

        fig = plt.gcf()

        # Force shared x-axis range if requested (affects the main data axis, not colorbar)
        if xlim is not None and fig.axes:
            try:
                # Prefer an axis that actually has plotted collections (beeswarm dots)
                main_ax = next(ax for ax in fig.axes if getattr(ax, "collections", []))
            except StopIteration:
                main_ax = fig.axes[0]
            main_ax.set_xlim(xlim)

        if fig.axes:
            # Title font size: prefer explicit fonts["title"] if present (already set via rc),
            # but allow overriding here too in case users only set subplot titles.
            title_fs = None
            if fonts and "title" in fonts:
                title_fs = fonts["title"] * float(font_scale)
            fig.axes[0].set_title(title, pad=10, fontsize=title_fs)

        # Enforce tick/label sizes across main axis & colorbar axis
        _apply_font_sizes_to_axes(
            fig,
            {
                k: (v * float(font_scale) if v is not None else None)
                for k, v in (fonts or {}).items()
            },
        )

        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=panel_dpi, bbox_inches="tight")
        plt.close(fig)

    buf.seek(0)
    return buf.getvalue()


def plot_stacked_beeswarms(
    shap_values_all,
    feature_sets,
    outdir="shap_plots",
    max_display=20,
    limit_wells=None,
    composite_fname="beeswarm_stacked.jpg",  # final JPEG file
    show=True,
    rng_seed=None,
    rng=None,
    final_dpi=600,
    panel_dpi=600,
    fonts=None,
    font_scale=1.0,
    subplot_label_fs=None,
    xlim=None,  # NEW: force x-axis range for all panels, e.g., (-1, 1)
):
    """
    Creates one figure with all requested feature-set beeswarm plots stacked vertically.
    Saves ONLY the final composite as a JPEG at final_dpi.

    Reproducibility:
    - Provide rng as an np.random.Generator, OR
    - Provide rng_seed (int) to create one internally.

    Font control:
    - Pass a dict fonts with any of: base, title, axis_label, tick, legend, suptitle, cbar_tick, cbar_label, subplot_label
    - Optionally multiply all via font_scale (float).

    xlim:
    - If provided, each individual beeswarm panel is rendered with these x-limits before stacking.
    """
    os.makedirs(outdir, exist_ok=True)

    # Build a generator if only a seed was given
    if rng is None and rng_seed is not None:
        rng = np.random.default_rng(rng_seed)

    # Pre-spawn child RNGs for each feature set (if rng provided)
    child_rngs = None
    if rng is not None:
        # Derive a stable base seed from the generator state
        state = rng.bit_generator.state
        try:
            base_seed = state["state"]["state"]
        except Exception:
            base_seed = abs(hash(repr(state))) % (2**32 - 1)
        ss = np.random.SeedSequence(base_seed)
        child_rngs = [np.random.default_rng(s) for s in ss.spawn(len(feature_sets))]

    # Collect per-panel PNG rasters in memory
    png_images = []
    used_names = []

    for i, fs_name in enumerate(feature_sets):
        out = _concat_shap_for_feature_set(shap_values_all, fs_name, limit_wells=limit_wells)
        if out[0] is None:
            print(f"[SHAP] No SHAP data for feature set '{fs_name}'. Skipping.")
            continue

        shap_mat, features_df, feature_names = out
        rng_i = child_rngs[i] if child_rngs else rng

        png_bytes = _render_beeswarm_to_png_bytes(
            shap_mat,
            features_df,
            feature_names,
            title=f"{fs_name}",
            max_display=max_display,
            rng=rng_i,
            panel_dpi=panel_dpi,
            fonts=fonts,
            font_scale=font_scale,
            xlim=xlim,  # ensure identical x-range across panels
        )
        png_images.append(plt.imread(io.BytesIO(png_bytes), format="png"))
        used_names.append(fs_name)

    if not png_images:
        print("[SHAP] No SHAP data found for any requested feature set.")
        return

    # Compute figure size from pixel dims / panel_dpi so text scales well
    n = len(png_images)
    fig_height = sum(img.shape[0] for img in png_images) / float(panel_dpi)
    fig_width = max(img.shape[1] for img in png_images) / float(panel_dpi)

    fig, axes = plt.subplots(n, 1, figsize=(fig_width, fig_height), constrained_layout=True)
    if n == 1:
        axes = [axes]  # normalize to list

    # Subplot label fontsize: explicit arg -> fonts dict -> global default
    if subplot_label_fs is None:
        if fonts and ("subplot_label" in fonts):
            subplot_label_fs = fonts["subplot_label"] * float(font_scale)
        else:
            subplot_label_fs = SUBPLOT_LABEL_FS * float(font_scale)

    for idx, (ax, img) in enumerate(zip(axes, png_images)):
        ax.imshow(img, zorder=0)
        ax.axis("off")
        # Subplot label: (a), (b), ... placed INSIDE the panel (top-left)
        ax.text(
            0.015,
            0.985,
            _index_to_alpha_label(idx),
            transform=ax.transAxes,
            fontsize=subplot_label_fs,
            fontweight="bold",
            va="top",
            ha="left",
            zorder=5,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", boxstyle="round,pad=0.15"),
        )

    outpath = os.path.join(outdir, composite_fname)
    # Save ONLY the final composite as JPEG @ specified dpi
    fig.savefig(outpath, format="jpeg", dpi=final_dpi, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    print(f"[SHAP] Saved stacked beeswarm JPEG to: {outpath}")


# -------- choose feature sets to stack (edit this list) --------
compare_feature_sets = ["no_seasonal", "with_seasonal"]

# Fallback to any two that exist in the SHAP store
available_fs = sorted({fs for (_, fs) in shap_values_all.keys()})
selected = [fs for fs in compare_feature_sets if fs in available_fs]
if len(selected) < 2:
    selected = available_fs[:2]

if len(selected) < 2:
    print("Need at least two feature sets with SHAP data.")
else:
    # Example font overrides (edit freely). Omit keys you don't need.
    my_fonts = {
        "base": 11,         # global default text size
        "title": 14,        # per-panel title ("SHAP Beeswarm — ...")
        "axis_label": 14,   # "SHAP value" / any axis labels
        "tick": 13,         # x ticks and y (feature) ticks
        "legend": 11,       # (not used by beeswarm, but harmless)
        "cbar_tick": 13,    # colorbar tick labels
        "cbar_label": 18,   # colorbar label ("Feature value")
        "suptitle": 16,     # (reserved if you add a suptitle later)
        "subplot_label": 16 # "(a)", "(b)" labels on stacked panels
    }

    plot_stacked_beeswarms(
        shap_values_all,
        feature_sets=selected,           # stacked into one figure
        outdir="shap_plots",
        max_display=20,                  # top N features per beeswarm
        limit_wells=None,                # optionally cap number of wells aggregated
        composite_fname="beeswarm_stacked.jpg",  # final JPEG
        show=True,
        rng_seed=42,                     # deterministic jitter; silences FutureWarning
        final_dpi=600,                   # JPEG dpi
        panel_dpi=600,                   # rasterize panels at high dpi
        fonts=my_fonts,                  # full control here
        font_scale=1.3,                  # multiply all sizes at once (e.g., 1.25)
        xlim=(-1, 1),                    # NEW: force identical x-axis across panels
        # subplot_label_fs=18            # optional explicit override
    )


In [ ]:
# save the imputed gw_level data
df_imputed_vals = pd.concat(
    [df for imputed_list in imputed_all_models.values() for df in imputed_list],
    ignore_index=True
)

# Show unique values
print("Unique well_no values:", df_imputed_vals['well_no'].unique())
print("Unique feature_set values:", df_imputed_vals['feature_set'].unique())
df_imputed_vals.head()

In [ ]:
# Select and rename the relevant columns from df_imputed
selected_cols = df_imputed_vals[['well_no', 'date', 'gw_level_m_asl']].rename(
    columns={'gw_level_m_asl': 'gw_level_m_asl_imputed'}
)

# Perform a left join with df_work on 'well_no' and 'date'
df_merged = df_work.merge(selected_cols, on=['well_no', 'date'], how='left')

# Display the result
df_merged.head()

# Save the result as a CSV file in Colab
df_merged.to_csv('/content/imputed_gw_level.csv', index=False)

## Groundwater level signal noise evaluation

This analysis pipeline aims to test the hypothesis that wells with noisier groundwater level time series (as measured by std_first_diff) tend to have poorer machine learning model performance (lower cross-validated R² scores). By combining model CV results with noise metrics and visualizing their correlation, the code helps quantify how signal quality impacts the reliability of groundwater predictions.

In [ ]:
# Sort by well and date
df_sorted = df_imputed_vals.sort_values(['well_no', 'date'])

# Filter LT-220 data to include only records till 2014 onwards. Seem like signal became more noisy.
# df_sorted = df_sorted[~((df_sorted['well_no'] == 'LT_220') & (df_sorted['date'] > '2014-01-01'))]

df_sorted['gw_level_zscore'] = df_sorted.groupby('well_no')['gw_level_m_asl'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Calculate basic noise metrics per well
noise_summary = df_sorted.groupby('well_no')['gw_level_zscore'].agg(
    std_dev='std',                     # Standard deviation
    mean='mean'                       # Mean (for CV calculation)
).reset_index()

# Add Coefficient of Variation (CV)
noise_summary['cv'] = noise_summary['std_dev'] / noise_summary['mean']

# Calculate first differences per well
df_sorted['first_diff'] = df_sorted.groupby('well_no')['gw_level_zscore'].diff()
first_diff_std = df_sorted.groupby('well_no')['first_diff'].std().reset_index(name='std_first_diff')

# Merge into noise summary
noise_summary = noise_summary.merge(first_diff_std, on='well_no')
noise_summary

In [ ]:
best_model_per_well
best_model_params
best_params

In [ ]:
# --- Correlation Plot (exclude AVERAGE row so it doesn't skew regression) ---
plot_df = summary[summary["well_no"] != "AVERAGE"].copy()

plot_df = plot_df.merge(
    noise_summary,
    on='well_no',
    how='left',
)

# Font sizes
TITLE_FS = 20
LABEL_FS = 16
TICK_FS  = 14
LEGEND_FS = 14

plt.figure(figsize=(8, 6))
ax = sns.regplot(
    data=plot_df,
    x="std_first_diff",
    y="CV_R2 mean",
    scatter_kws={"s": 50, "alpha": 0.7},
    line_kws={"linewidth": 2.2}
)

# ax.set_title("Noise vs Cross-Validated Performance", fontsize=TITLE_FS, pad=12)
ax.set_xlabel("Groundwater-level noise (std of first differences)", fontsize=LABEL_FS)
ax.set_ylabel("Mean CV R\u00b2", fontsize=LABEL_FS)

ax.tick_params(axis="both", labelsize=TICK_FS)
ax.grid(True, which="major", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("gw_level_noise.jpeg", dpi=600, bbox_inches="tight", pad_inches=0.2)
plt.show()

## VIF Table

Variance Inflation Factor (VIF) is a key diagnostic metric used to detect multicollinearity among predictor variables in a regression or predictive modeling context. Multicollinearity occurs when two or more features are highly correlated, causing redundancy and instability in the model coefficients.

Why Multicollinearity Matters:

* Inflated standard errors, leading to less reliable coefficient estimates.

* Difficulty in interpreting individual feature effects.

* Reduced model generalizability and poorer predictive performance.

What VIF Measures:

* VIF quantifies how much the variance of a regression coefficient is inflated due to collinearity with other features.

* A VIF of 1 means no correlation; values above 1 indicate increasing correlation with other variables.

Why Select Feature Sets with VIF < 5?
* A common rule of thumb is to keep VIF values below 5 (some use 10 as a cutoff) to ensure multicollinearity is at an acceptable level.

* Features with VIF ≥ 5 are likely highly collinear with others, suggesting redundant information that can degrade model quality.

By selecting features with VIF < 5 in feature sets above:

* You reduce redundancy and noise.

* Improve model stability and interpretability.

* Help prevent overfitting by avoiding highly correlated predictors.

In [ ]:
def compute_avg_vif_table(vif_all):
    """
    Compute average and standard deviation of VIF scores across wells for each feature set.

    Parameters:
    - vif_all (dict): {feature_set_name: [ {feature: vif, ...}, ... ]}

    Returns:
    - avg_vif_df: pd.DataFrame with MultiIndex columns:
        (feature_set, 'mean') and (feature_set, 'std'),
      where features are rows.
    """
    avg_vif_data = {}

    for feature_set_name, vif_dicts in vif_all.items():
        if not vif_dicts:
            continue

        # Convert list of dicts to DataFrame
        vif_df = pd.DataFrame(vif_dicts)

        # Compute mean and std
        avg_vif = vif_df.mean().round(1)
        std_vif = vif_df.std().round(1)

        # Store results as tuples for MultiIndex columns
        avg_vif_data[(feature_set_name, 'mean')] = avg_vif
        avg_vif_data[(feature_set_name, 'std')] = std_vif

    # Combine into a single DataFrame
    avg_vif_df = pd.DataFrame(avg_vif_data)

    # Sort MultiIndex for cleaner output
    avg_vif_df = avg_vif_df.sort_index(axis=1, level=0)

    return avg_vif_df


# Example usage
avg_vif_df = compute_avg_vif_table(vif_all)
avg_vif_df.to_csv('avg_vif_imput.csv')
avg_vif_df

## Feature Importance Plots

Across both model comparisons, groundwater storage (gws_mm_tavg_gldas) consistently emerges as one of the strongest predictors of groundwater level (gw_level_m_asl). Its high mean feature importance and relatively low variability suggest that:

* GWS carries critical historical memory of the groundwater system, reflecting the cumulative effects of recharge and discharge processes.

* Incorporating GWS helps models capture underlying trends and improve prediction accuracy beyond immediate meteorological inputs.

In [ ]:
# Select up to two model-feature set combos to compare
plot_keys = list(feature_importances_all.keys())[:2]

if len(plot_keys) < 2:
    print("Need at least two models to plot one-above-the-other.")
else:
    # Font sizes
    TITLE_FS = 18
    LABEL_FS = 15
    TICK_FS  = 14
    SUBPLOT_LABEL_FS = 16

    # Create vertical subplots with a shared y-axis scale
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharey=True)

    # Ensure axes is iterable even if matplotlib returns a single Axes
    if not isinstance(axes, (list, np.ndarray)):
        axes = [axes]

    for idx, key in enumerate(plot_keys):
        model_name, feature_set_name = key
        df_feat = pd.DataFrame(feature_importances_all[key]).fillna(0)

        # Compute mean and std of feature importances across wells
        mean_importance = df_feat.mean().sort_values(ascending=False)
        std_importance  = df_feat.std().reindex(mean_importance.index)

        ax = axes[idx]
        positions = range(len(mean_importance))

        # Bar plot with error bars
        ax.bar(positions, mean_importance.values, yerr=std_importance.values, capsize=5)

        # X ticks and rotated labels (bigger font)
        ax.set_xticks(positions)
        ax.set_xticklabels(mean_importance.index, rotation=45, ha='right', fontsize=TICK_FS)

        # Y ticks font
        ax.tick_params(axis='y', labelsize=TICK_FS)

        # Horizontal grid lines
        ax.grid(axis='y', linestyle='--', alpha=0.6)

        # Title and y-label (only top plot to avoid redundancy)
        # ax.set_title(f"{model_name} | {feature_set_name}", fontsize=TITLE_FS, pad=10)
        if idx == 0:
            ax.set_ylabel("Mean Feature Importance", fontsize=LABEL_FS)

        # Subplot label (a), (b), etc.
        label = f"({chr(97 + idx)})"
        ax.text(0.01, 1.02, label, transform=ax.transAxes,
                fontsize=SUBPLOT_LABEL_FS, fontweight='bold',
                va='bottom', ha='left', clip_on=False)

    # Adjust layout to prevent overlap with rotated x labels
    plt.tight_layout()
    plt.subplots_adjust(hspace=1.10, bottom=0.18)

    # Save high-res figure
    plt.savefig("feature_importance_comparison_imput.jpeg", dpi=600, bbox_inches='tight')

    # Show plot
    plt.show()

## GW Level Timeseries

In [ ]:
import matplotlib.dates as mdates
from collections import OrderedDict

# --- Print-ready sizing (A4 portrait) and larger fonts ---
A4_W_IN, A4_H_IN = 8.27, 11.69   # inches
DPI = 300                        # print quality

plt.rcParams.update({
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "axes.titlesize": 14,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
})

# --- Gather all plotting data (unchanged) ---
plots = []

for key, data_list in imputed_all_models.items():
    model_name, feature_set_name = key
    if not data_list:
        continue

    df_imputed = pd.concat(data_list).sort_values(by=['well_no', 'date'])

    for well_no in df_imputed['well_no'].unique():
        imputed = df_imputed[df_imputed['well_no'] == well_no].sort_values('date')

        actual = df_work[df_work['well_no'] == well_no].copy()
        actual['date'] = pd.to_datetime(actual['date'])
        actual['was_originally_missing'] = actual[target_col].isnull()

        merged = pd.merge(imputed, actual[['date', 'was_originally_missing']], on='date', how='left')
        observed = merged[~merged['was_originally_missing']].copy()
        imputed_only = merged[merged['was_originally_missing']].copy()

        # Segment observed values
        observed = observed[['date', target_col]].sort_values('date')
        observed['gap'] = observed['date'].diff().dt.days.fillna(1)
        observed['segment'] = (observed['gap'] > 1).cumsum()

        # Get masked intervals
        masked_segments = cv_masked_intervals.get((well_no, model_name, feature_set_name), [])

        plots.append((well_no, model_name, feature_set_name, observed, imputed_only, masked_segments))

# --- Create A4 portrait figure with stacked subplots and shared x-axis ---
n_plots = len(plots)
if n_plots == 0:
    raise ValueError("No plots to draw.")

fig, axes = plt.subplots(
    n_plots, 1,
    figsize=(A4_W_IN, A4_H_IN),
    sharex=True
)

if n_plots == 1:
    axes = [axes]  # ensure iterable

# Date formatting for the shared x-axis
locator   = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)

# Collect legend entries once, then show a single legend at the bottom
legend_pairs = []  # list of (handle, label)

for idx, (ax, plot_data) in enumerate(zip(axes, plots)):
    well_no, model_name, feature_set_name, observed, imputed_only, masked_segments = plot_data

    # Plot observed line segments
    for i, seg in observed.groupby('segment'):
        h_obs = ax.plot(seg['date'], seg[target_col],
                        color='black', linewidth=2.4,
                        label='Observed' if i == 0 else '')
        if i == 0:
            legend_pairs.append((h_obs[0], 'Observed'))

    # Plot imputed values
    if not imputed_only.empty:
        h_imp = ax.scatter(imputed_only['date'], imputed_only[target_col],
                           s=22, color='tab:blue', label='Imputed')
        legend_pairs.append((h_imp, 'Imputed'))

    # Plot masked intervals
    for j, (start_date, end_date) in enumerate(masked_segments):
        h_span = ax.axvspan(start_date, end_date, color='red', alpha=0.2,
                            label='CV Masked Interval' if j == 0 else '')
        if j == 0:
            legend_pairs.append((h_span, 'CV Masked Interval'))

    # Subplot label (a), (b), ...
    label = f"({chr(97 + idx)})"
    ax.text(0.01, 1.02, label, transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='bottom', ha='left', clip_on=False)

    # Title, grid (horizontal + vertical), and tidy margins
    ax.set_title(f"Well {well_no}", pad=6)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.grid(axis='x', linestyle=':',  alpha=0.5)
    ax.margins(x=0.01)

    # Only bottom subplot shows x tick labels to save space
    if idx < n_plots - 1:
        ax.tick_params(axis='x', which='both', labelbottom=False)

# Shared axes cosmetics
axes[-1].xaxis.set_major_locator(locator)
axes[-1].xaxis.set_major_formatter(formatter)
axes[-1].set_xlabel("Date")

# Single shared Y label for all plots
fig.supylabel("GW Level (m ASL)", x=0.01, fontsize=16)

# Build a unique, ordered legend and place it below all subplots
unique = OrderedDict()
for h, lab in legend_pairs:
    if lab and lab not in unique:
        unique[lab] = h

# Reserve space at the bottom for the legend, then draw it
fig.tight_layout(rect=(0, 0.08, 1, 1))  # leave ~8% at bottom
fig.legend(list(unique.values()), list(unique.keys()),
           loc='lower center', ncol=3, frameon=True, bbox_to_anchor=(0.5, 0.01))

# Export at exact A4 portrait size
out_path = "stacked_imputation_timeseries.jpeg"
fig.savefig(out_path, bbox_inches='tight', pad_inches=0.3)
plt.show()